# R Analytics — NorthStar

Delivery, hub, driver, and complaint analysis using R.

## 1. Setup

In [ ]:
install.packages(c('dplyr', 'ggplot2', 'knitr', 'lubridate', 'scales'),
                 repos = 'https://cloud.r-project.org', quiet = TRUE)

library(dplyr); library(ggplot2); library(knitr); library(lubridate); library(scales)
cat('Packages loaded.\n')

In [ ]:
orders     <- read.csv('/content/orders_cleaned.csv',     stringsAsFactors = FALSE)
deliveries <- read.csv('/content/deliveries_cleaned.csv', stringsAsFactors = FALSE)
drivers    <- read.csv('/content/drivers_cleaned.csv',    stringsAsFactors = FALSE)
complaints <- read.csv('/content/complaints_cleaned.csv', stringsAsFactors = FALSE)
hubs       <- read.csv('/content/hubs_cleaned.csv',       stringsAsFactors = FALSE)
cat('Data loaded.\n')

## 2. Delivery performance

In [ ]:
outcome_summary <- deliveries %>%
  group_by(delivery_status) %>%
  summarise(count = n(), .groups = 'drop') %>%
  mutate(percent = round(100 * count / sum(count), 1))

kable(outcome_summary, caption = 'Delivery Outcome Distribution')

In [ ]:
ggplot(outcome_summary, aes(x = reorder(delivery_status, -count), y = count, fill = delivery_status)) +
  geom_col(show.legend = FALSE) +
  geom_text(aes(label = paste0(count, ' (', percent, '%)')), vjust = -0.4) +
  scale_fill_manual(values = c('OnTime' = 'steelblue', 'Late' = '#FCD34D', 'Failed' = 'tomato')) +
  labs(title = 'Delivery Outcome Distribution',
       subtitle = paste0('n = ', sum(outcome_summary$count), ' deliveries'),
       x = 'Outcome', y = 'Deliveries') +
  theme_minimal()

In [ ]:
outcome_by_service <- deliveries %>%
  inner_join(orders %>% select(order_id, service_type), by = 'order_id') %>%
  group_by(service_type, delivery_status) %>%
  summarise(count = n(), .groups = 'drop') %>%
  group_by(service_type) %>%
  mutate(percent = round(100 * count / sum(count), 1))

ggplot(outcome_by_service, aes(x = service_type, y = percent, fill = delivery_status)) +
  geom_col(position = 'stack') +
  scale_fill_manual(values = c('OnTime' = 'steelblue', 'Late' = '#FCD34D', 'Failed' = 'tomato')) +
  labs(title = 'Delivery Outcome by Service Type',
       x = 'Service Type', y = 'Percent (%)', fill = 'Outcome') +
  theme_minimal()

## 3. Hub and zone performance

In [ ]:
hub_perf <- deliveries %>%
  inner_join(hubs %>% select(hub_id, hub_name, zone), by = 'hub_id') %>%
  group_by(hub_name, zone) %>%
  summarise(
    total    = n(),
    failed   = sum(delivery_status == 'Failed'),
    fail_pct = round(100 * failed / total, 1),
    .groups  = 'drop'
  ) %>%
  arrange(desc(fail_pct))

kable(hub_perf, caption = 'Hub Performance')

In [ ]:
hub_long <- deliveries %>%
  inner_join(hubs %>% select(hub_id, hub_name), by = 'hub_id') %>%
  group_by(hub_name, delivery_status) %>%
  summarise(count = n(), .groups = 'drop') %>%
  group_by(hub_name) %>%
  mutate(percent = round(100 * count / sum(count), 1))

ggplot(hub_long, aes(x = reorder(hub_name, -percent * (delivery_status == 'Failed')),
                     y = percent, fill = delivery_status)) +
  geom_col() +
  scale_fill_manual(values = c('OnTime' = 'steelblue', 'Late' = '#FCD34D', 'Failed' = 'tomato')) +
  labs(title = 'Delivery Outcomes by Hub', x = 'Hub', y = 'Percent (%)', fill = 'Outcome') +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

In [ ]:
zone_pair <- deliveries %>%
  inner_join(orders %>% select(order_id, pickup_zone, dropoff_zone), by = 'order_id') %>%
  filter(delivery_status == 'Failed') %>%
  group_by(pickup_zone, dropoff_zone) %>%
  summarise(failed_count = n(), .groups = 'drop')

ggplot(zone_pair, aes(x = dropoff_zone, y = pickup_zone, fill = failed_count)) +
  geom_tile(colour = 'white') +
  geom_text(aes(label = failed_count), colour = 'black', size = 3.5) +
  scale_fill_gradient(low = '#FFE4E1', high = 'tomato') +
  labs(title = 'Failed Deliveries by Pickup x Dropoff Zone',
       x = 'Dropoff Zone', y = 'Pickup Zone', fill = 'Failed Count') +
  theme_minimal()

## 4. Driver analysis

In [ ]:
active_drivers <- drivers %>% filter(active_flag == 1, !is.na(driver_rating))
cor_exp <- cor(active_drivers$years_experience, active_drivers$driver_rating, use = 'complete.obs')

ggplot(active_drivers, aes(x = years_experience, y = driver_rating)) +
  geom_point(colour = 'steelblue', alpha = 0.6) +
  geom_smooth(method = 'lm', se = FALSE, colour = 'red') +
  labs(title = 'Driver Rating vs Years of Experience',
       subtitle = paste0('r = ', round(cor_exp, 3)),
       x = 'Years of Experience', y = 'Driver Rating') +
  theme_minimal()

In [ ]:
with_training <- drivers %>% filter(active_flag == 1, !is.na(driver_rating), !is.na(training_score))
cor_train <- cor(with_training$training_score, with_training$driver_rating, use = 'complete.obs')

ggplot(with_training, aes(x = training_score, y = driver_rating)) +
  geom_point(colour = 'steelblue', alpha = 0.6) +
  geom_smooth(method = 'lm', se = FALSE, colour = 'red') +
  labs(title = 'Driver Rating vs Training Score',
       subtitle = paste0('r = ', round(cor_train, 3)),
       x = 'Training Score', y = 'Driver Rating') +
  theme_minimal()

In [ ]:
ride_overrides <- deliveries %>%
  inner_join(drivers %>% select(driver_id, employment_type), by = 'driver_id')

ggplot(ride_overrides, aes(x = employment_type, y = manual_route_override_count, fill = employment_type)) +
  geom_boxplot(show.legend = FALSE) +
  labs(title = 'Manual Route Overrides by Employment Type',
       x = 'Employment Type', y = 'Overrides per Delivery') +
  theme_minimal()

In [ ]:
override_summary <- ride_overrides %>%
  group_by(employment_type) %>%
  summarise(
    n      = n(),
    mean   = round(mean(manual_route_override_count, na.rm = TRUE), 2),
    median = median(manual_route_override_count, na.rm = TRUE),
    iqr    = round(IQR(manual_route_override_count, na.rm = TRUE), 2),
    .groups = 'drop'
  )
kable(override_summary, caption = 'Override Statistics by Employment Type')

## 5. Complaint patterns

In [ ]:
complaints_monthly <- complaints %>%
  mutate(month = floor_date(as.Date(created_at), 'month')) %>%
  group_by(month) %>%
  summarise(complaint_count = n(), .groups = 'drop')

ggplot(complaints_monthly, aes(x = month, y = complaint_count)) +
  geom_line(colour = 'steelblue', linewidth = 1) +
  geom_point(colour = 'steelblue', size = 2.5) +
  scale_x_date(date_breaks = '2 months', date_labels = '%b %Y') +
  labs(title = 'Monthly Complaint Volume', x = 'Month', y = 'Complaints') +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

In [ ]:
severity_by_type <- complaints %>%
  mutate(severity = factor(severity, levels = c('Low', 'Medium', 'High'))) %>%
  group_by(complaint_type, severity) %>%
  summarise(count = n(), .groups = 'drop') %>%
  group_by(complaint_type) %>%
  mutate(percent = round(100 * count / sum(count), 1))

ggplot(severity_by_type, aes(x = complaint_type, y = percent, fill = severity)) +
  geom_col() +
  scale_fill_manual(values = c('Low' = '#86EFAC', 'Medium' = '#FCD34D', 'High' = 'tomato')) +
  labs(title = 'Severity by Complaint Type',
       x = 'Complaint Type', y = 'Percent (%)', fill = 'Severity') +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))